In [19]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/player-intelligence-system-cpe-342-ml/cpe342-kaggle-main/sample_submission.csv
/kaggle/input/competitions/player-intelligence-system-cpe-342-ml/cpe342-kaggle-main/README.md
/kaggle/input/competitions/player-intelligence-system-cpe-342-ml/cpe342-kaggle-main/03_regression/train.csv
/kaggle/input/competitions/player-intelligence-system-cpe-342-ml/cpe342-kaggle-main/03_regression/test.csv
/kaggle/input/competitions/player-intelligence-system-cpe-342-ml/cpe342-kaggle-main/03_regression/datacard.txt
/kaggle/input/competitions/player-intelligence-system-cpe-342-ml/cpe342-kaggle-main/02_multiclass_classification/train.csv
/kaggle/input/competitions/player-intelligence-system-cpe-342-ml/cpe342-kaggle-main/02_multiclass_classification/test.csv
/kaggle/input/competitions/player-intelligence-system-cpe-342-ml/cpe342-kaggle-main/02_multiclass_classification/datacard.txt
/kaggle/input/competitions/player-intelligence-system-cpe-342-ml/cpe342-kaggle-main/04_image_classifica

In [20]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

In [21]:
KAGGLE_DIR = ('/kaggle/input/competitions/player-intelligence-system-cpe-342-ml/'
              'cpe342-kaggle-main/03_regression')
LOCAL_DIR = '/mnt/user-data/uploads'
DATA_DIR = KAGGLE_DIR if os.path.isdir(KAGGLE_DIR) else LOCAL_DIR

OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/mnt/user-data/outputs'
OUT = os.path.join(OUT_DIR, 'submission.csv')
os.makedirs(OUT_DIR, exist_ok=True)

train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

# ค้นหาไฟล์ submission ตั้งต้น (ตรวจสอบทุกโฟลเดอร์ใน /kaggle/input และโฟลเดอร์ปัจจุบัน)
sub_candidates = [
    os.path.join(DATA_DIR, 'submission_final (5).csv'),
    os.path.join(OUT_DIR, 'submission_final (5).csv'),
    'submission_final (5).csv',
] + sorted(glob.glob('/kaggle/input/**/*submission*.csv', recursive=True))

sub_path = next((p for p in sub_candidates if os.path.isfile(p) and 'sample' not in p.lower()), None)
if sub_path is None:
    raise FileNotFoundError("ไม่พบไฟล์ submission_final (5).csv กรุณาตรวจสอบชื่อไฟล์หรืออัปโหลดไฟล์เข้า Dataset ก่อนรัน")

print(f"โหลดไฟล์ submission ตั้งต้นจาก: {sub_path}")
sub = pd.read_csv(sub_path)
assert len(sub) == len(test), f"จำนวนแถวไม่ตรงกัน: sub={len(sub)}, test={len(test)}"

โหลดไฟล์ submission ตั้งต้นจาก: /kaggle/input/datasets/rattananansiriponwat/submission-final-5/submission_final (5).csv


In [22]:
def build_features(df, ref_train=None):  # <--- เพิ่ม , ref_train=None ในวงเล็บตรงนี้
    X = df.drop(columns=['player_id', 'spending_30d'], errors='ignore').copy()
    
    # เติมค่าว่างจากความสัมพันธ์เชิงคณิตศาสตร์ที่แม่นยำในข้อมูล
    mask_hist = X['historical_spending'].isna() & X['total_transactions'].notna() & X['avg_transaction_value'].notna()
    X.loc[mask_hist, 'historical_spending'] = np.maximum(X.loc[mask_hist, 'total_transactions'], 1) * X.loc[mask_hist, 'avg_transaction_value']
    
    mask_avg = X['avg_transaction_value'].isna() & X['historical_spending'].notna() & X['total_transactions'].notna()
    X.loc[mask_avg, 'avg_transaction_value'] = X.loc[mask_avg, 'historical_spending'] / np.maximum(X.loc[mask_avg, 'total_transactions'], 1)
    
    # ฟีเจอร์เชิงพฤติกรรมและการใช้จ่าย
    X['missing_count'] = X.isna().sum(axis=1)
    X['hist_to_prev_ratio'] = X['historical_spending'] / (X['prev_month_spending'] + 1e-5)
    X['recent_spend_momentum'] = X['prev_month_spending'] / ((X['historical_spending'] / 3.0) + 1e-5)
    X['spend_per_day_age'] = X['historical_spending'] / (X['account_age_days'] + 1.0)
    X['spend_per_playtime'] = X['historical_spending'] / (X['total_playtime_hours'] + 1.0)
    X['weekly_play_minutes'] = X['sessions_per_week'] * X['avg_session_length']
    X['discount_tx_ratio'] = X['purchases_on_discount'] / np.maximum(X['total_transactions'], 1.0)
    X['recency_ratio'] = X['days_since_last_login'] / (X['days_since_last_purchase'] + 1.0)
    X['vip_x_prev_spend'] = (X['vip_status'] + 1) * X['prev_month_spending']
    X['event_x_prev_spend'] = X['event_participation_rate'] * X['prev_month_spending']
    
    # ฟีเจอร์เปรียบเทียบกับค่ากลางของกลุ่มตัวเอง
    if ref_train is not None:
        for grp in ['segment', 'vip_status']:
            grp_median = ref_train.groupby(grp)['prev_month_spending'].median()
            X[f'prev_spend_diff_{grp}'] = X['prev_month_spending'] - X[grp].map(grp_median)
            X[f'prev_spend_ratio_{grp}'] = X['prev_month_spending'] / (X[grp].map(grp_median) + 1.0)

    X['prev_share_of_hist'] = X['prev_month_spending'] / (X['historical_spending'] + 1.0)
    X['net_spend_after_discount'] = X['historical_spending'] * (1.0 - X['discount_rate_used'].fillna(0))
    
    for col in ['historical_spending', 'prev_month_spending', 'avg_transaction_value']:
        X[f'log_{col}'] = np.log1p(np.maximum(X[col], 0))
        
    return X

# <--- ส่ง ref_train=train เข้าไปตอนเรียกใช้ทั้ง 2 บรรทัดนี้ด้วยครับ
X_train = build_features(train, ref_train=train)
y_train = train['spending_30d'].values
X_test = build_features(test, ref_train=train)

In [23]:
N_SPLITS = 7
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
quantiles = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.55]

oof_prob = np.zeros(len(train))
test_prob = np.zeros(len(test))

oof_q = {q: np.zeros(len(train)) for q in quantiles}
test_q = {q: np.zeros(len(test)) for q in quantiles}

oof_raw_pos = np.zeros(len(train))
test_raw_pos = np.zeros(len(test))

oof_1stage = np.zeros(len(train))
test_1stage = np.zeros(len(test))

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[tr_idx], y_train[tr_idx]
    X_va, y_va = X_train.iloc[va_idx], y_train[va_idx]
    pos_mask = y_tr > 0
    
    # --- 1) Classifier: ทำนายโอกาสการซื้อ P(y > 0) ---
    clf = HistGradientBoostingClassifier(
          max_iter=1000, learning_rate=0.03, max_leaf_nodes=95, min_samples_leaf=45,
          l2_regularization=2.0, categorical_features=cat_cols, random_state=42 + fold
    )
    clf.fit(X_tr, (y_tr > 0).astype(int))
    p_va = clf.predict_proba(X_va)[:, 1]
    p_te = clf.predict_proba(X_test)[:, 1]
    
    if HAS_LGB:
        lgb_clf = lgb.LGBMClassifier(
            n_estimators=1000, learning_rate=0.03, num_leaves=63, min_child_samples=30,
            subsample_for_bin=50000, colsample_bytree=0.8, subsample=0.8, random_state=42 + fold, verbose=-1
        )
        lgb_clf.fit(X_tr, (y_tr > 0).astype(int), categorical_feature=cat_cols)
        p_va = 0.5 * p_va + 0.5 * lgb_clf.predict_proba(X_va)[:, 1]
        p_te = 0.5 * p_te + 0.5 * lgb_clf.predict_proba(X_test)[:, 1]
        
    oof_prob[va_idx] = p_va
    test_prob += p_te / N_SPLITS
    
    # --- 2) Multi-Quantile Regressor บน log1p(y) สำหรับกลุ่มที่ซื้อ ---
    for q in quantiles:
        rq = HistGradientBoostingRegressor(
             loss='quantile', quantile=q, max_iter=1000, learning_rate=0.03,
             max_leaf_nodes=95, min_samples_leaf=40, l2_regularization=2.0,
             categorical_features=cat_cols, random_state=42 + fold
        )
        rq.fit(X_tr[pos_mask], np.log1p(y_tr[pos_mask]))
        pred_va_q = np.clip(np.expm1(rq.predict(X_va)), 0, 300000)
        pred_te_q = np.clip(np.expm1(rq.predict(X_test)), 0, 300000)
        
        if HAS_LGB and q == 0.50:
            lgb_med = lgb.LGBMRegressor(
                objective='regression_l1', n_estimators=1200, learning_rate=0.03, num_leaves=63,
                min_child_samples=25, colsample_bytree=0.8, subsample=0.8, random_state=42 + fold, verbose=-1
            )
            lgb_med.fit(X_tr[pos_mask], np.log1p(y_tr[pos_mask]), categorical_feature=cat_cols)
            pred_va_q = 0.5 * pred_va_q + 0.5 * np.clip(np.expm1(lgb_med.predict(X_va)), 0, 300000)
            pred_te_q = 0.5 * pred_te_q + 0.5 * np.clip(np.expm1(lgb_med.predict(X_test)), 0, 300000)
            
        oof_q[q][va_idx] = pred_va_q
        test_q[q] += pred_te_q / N_SPLITS
        
    # --- 3) Raw-Scale L1 Regressor สำหรับกลุ่มที่ซื้อ (เก็บความแม่นยำกลุ่ม Whales) ---
    reg_raw = HistGradientBoostingRegressor(
        loss='absolute_error', max_iter=1000, learning_rate=0.03, max_leaf_nodes=95,
        min_samples_leaf=40, categorical_features=cat_cols, random_state=42 + fold
    )
    reg_raw.fit(X_tr[pos_mask], y_tr[pos_mask])
    oof_raw_pos[va_idx] = np.clip(reg_raw.predict(X_va), 0, 300000)
    test_raw_pos += np.clip(reg_raw.predict(X_test), 0, 300000) / N_SPLITS
    
    # --- 4) 1-Stage Raw-Scale L1 Regressor (เทรนรวมทั้งหมดเพื่อลด Variance) ---
    reg_1s = HistGradientBoostingRegressor(
        loss='absolute_error', max_iter=1000, learning_rate=0.03, max_leaf_nodes=95,
        min_samples_leaf=40, categorical_features=cat_cols, random_state=42 + fold
    )
    reg_1s.fit(X_tr, y_tr)
    oof_1stage[va_idx] = np.clip(reg_1s.predict(X_va), 0, 300000)
    test_1stage += np.clip(reg_1s.predict(X_test), 0, 300000) / N_SPLITS
    
    print(f"Fold {fold + 1}/{N_SPLITS} เสร็จสมบูรณ์")

Fold 1/7 เสร็จสมบูรณ์
Fold 2/7 เสร็จสมบูรณ์
Fold 3/7 เสร็จสมบูรณ์
Fold 4/7 เสร็จสมบูรณ์
Fold 5/7 เสร็จสมบูรณ์
Fold 6/7 เสร็จสมบูรณ์
Fold 7/7 เสร็จสมบูรณ์


In [24]:
def bayes_quantile_predict(probs, q_dict, th=0.50):
    max_q = max(quantiles)
    q_grid = np.array([0.0] + quantiles)
    q_mat = np.vstack([np.zeros_like(probs)] + [q_dict[q] for q in quantiles])
    target_q = np.clip(1.0 - th / np.maximum(probs, 1e-5), 0.0, max_q)
    
    out = np.zeros_like(probs)
    active = probs > th
    for i in np.where(active)[0]:
        out[i] = np.interp(target_q[i], q_grid, q_mat[:, i])
    return out

def calc_nmae(y, yhat):
    return 1.0 / (1.0 + np.sum(np.abs(y - yhat)) / np.sum(y))

# 1. ค้นหาค่า Threshold (best_th) ที่ให้ค่า NMAE สูงที่สุดบน OOF
best_th = 0.50
best_q_score = -1.0
for th in np.linspace(0.42, 0.58, 33):
    score = calc_nmae(y_train, bayes_quantile_predict(oof_prob, oof_q, th=th))
    if score > best_q_score:
        best_q_score = score
        best_th = th

print(f"Best Quantile Threshold: {best_th:.4f} (Single BayesQuantile OOF NMAE: {best_q_score:.6f})")

# 2. สร้างคำทำนาย OOF โดยใช้ best_th แทนค่าตายตัว 0.50
oof_bayes = bayes_quantile_predict(oof_prob, oof_q, th=best_th)
oof_raw_hurdle = np.where(oof_prob >= best_th, oof_raw_pos, 0.0)

# 3. หาสัดส่วนการผสม (Ensemble Weights) ที่ดีที่สุดบน OOF
best_score = -1.0
best_w = (0.5, 0.3, 0.2)
for w1 in np.linspace(0.3, 0.7, 9):
    for w2 in np.linspace(0.1, 0.5, 9):
        w3 = round(1.0 - w1 - w2, 4)
        if w3 < 0:
            continue
        blend = w1 * oof_bayes + w2 * oof_raw_hurdle + w3 * oof_1stage
        score = calc_nmae(y_train, blend)
        if score > best_score:
            best_score = score
            best_w = (w1, w2, w3)

print(f"Best Ensemble Weights (BayesQuantile, RawHurdle, 1Stage): {best_w}")
print(f"OOF NMAE: {best_score:.6f}")

# 4. สร้างคำทำนายชุดทดสอบโดยใช้ best_th และ best_w
test_bayes = bayes_quantile_predict(test_prob, test_q, th=best_th)
test_raw_hurdle = np.where(test_prob >= best_th, test_raw_pos, 0.0)

final_preds = best_w[0] * test_bayes + best_w[1] * test_raw_hurdle + best_w[2] * test_1stage
final_preds = np.round(np.clip(final_preds, 0, 300000), 2)

sub['task3'] = final_preds
sub.to_csv(OUT, index=False)
print(f"บันทึกไฟล์ผลลัพธ์เรียบร้อยที่: {OUT}")
print(sub.head(10))

Best Quantile Threshold: 0.4800 (Single BayesQuantile OOF NMAE: 0.789633)
Best Ensemble Weights (BayesQuantile, RawHurdle, 1Stage): (np.float64(0.7), np.float64(0.2), np.float64(0.1))
OOF NMAE: 0.789577
บันทึกไฟล์ผลลัพธ์เรียบร้อยที่: /kaggle/working/submission.csv
         id  task1  task2    task3  task4  task5
0  ANS00001      1      2     1.43      2      1
1  ANS00002      1      1  2296.30      2      0
2  ANS00003      1      3     0.00      3      1
3  ANS00004      1      2     2.96      0      0
4  ANS00005      0      0     5.19      3      0
5  ANS00006      1      0     1.60      2      0
6  ANS00007      1      1     0.00      4      0
7  ANS00008      1      1  3550.64      1      0
8  ANS00009      1      3     1.47      1      0
9  ANS00010      1      2   135.61      3      0
